# Problem 3 — Circuit Board Alignment and Comparison

Images `c1.jpg` and `c2.jpg` show the same circuit board from slightly different perspectives.

| Part | Task |
|------|------|
| (a) | Select ~6 corresponding point pairs manually, compute homography, warp c1 → c2 perspective |
| (b) | Subtract warped image from c2 to highlight differences |
| (c) | Detect and display SIFT/ORB keypoint matches between c1 and c2 |
| (d) | Compute homography from feature matches; compare with manual result |

In [ ]:
import cv2 as cv
import numpy as np
import matplotlib.pyplot as plt

c1 = cv.imread('../data/c1.jpg')
c2 = cv.imread('../data/c2.jpg')
c1_rgb = cv.cvtColor(c1, cv.COLOR_BGR2RGB)
c2_rgb = cv.cvtColor(c2, cv.COLOR_BGR2RGB)

print('c1 shape:', c1.shape)
print('c2 shape:', c2.shape)

fig, ax = plt.subplots(1, 2, figsize=(14, 6))
ax[0].imshow(c1_rgb); ax[0].set_title('c1'); ax[0].axis('off')
ax[1].imshow(c2_rgb); ax[1].set_title('c2'); ax[1].axis('off')
plt.tight_layout()
plt.show()

## Part (a) — Manual Homography

Click ~6 corresponding points in c1 and c2, compute the homography $H$ such that $c1 \xrightarrow{H} c2$, then warp c1 into c2's perspective.

In [ ]:
# --- Interactive point selection ---
# Uncomment and run in a non-inline backend to click corresponding points.

# %matplotlib tk
#
# def select_points(img_rgb, title, n=6):
#     fig, ax = plt.subplots(figsize=(12, 8))
#     ax.imshow(img_rgb)
#     ax.set_title(title)
#     pts = np.array(plt.ginput(n, timeout=60))
#     plt.close()
#     return pts
#
# pts_c1 = select_points(c1_rgb, 'Click 6 points in c1')
# pts_c2 = select_points(c2_rgb, 'Click matching 6 points in c2')
# print('c1 points:', pts_c1)
# print('c2 points:', pts_c2)

# --- Or paste pre-measured coordinates here ---
pts_c1 = np.array([   # (x, y) pixel coordinates in c1
    # [x1, y1],
    # [x2, y2],
    # ...
], dtype=np.float32)

pts_c2 = np.array([   # corresponding (x, y) pixel coordinates in c2
    # [x1, y1],
    # [x2, y2],
    # ...
], dtype=np.float32)

In [ ]:
# Compute homography from manually selected points
H_manual, mask_manual = cv.findHomography(pts_c1, pts_c2, method=0)
print('Manual Homography H:')
print(H_manual)

# Warp c1 into c2's perspective
h, w = c2.shape[:2]
c1_warped = cv.warpPerspective(c1, H_manual, (w, h))
c1_warped_rgb = cv.cvtColor(c1_warped, cv.COLOR_BGR2RGB)

fig, ax = plt.subplots(1, 3, figsize=(18, 6))
ax[0].imshow(c1_rgb);         ax[0].set_title('c1 (original)');  ax[0].axis('off')
ax[1].imshow(c1_warped_rgb);  ax[1].set_title('c1 warped → c2'); ax[1].axis('off')
ax[2].imshow(c2_rgb);         ax[2].set_title('c2 (target)');    ax[2].axis('off')
plt.tight_layout()
plt.savefig('../output/p3a_manual_warp.png', dpi=150)
plt.show()

## Part (b) — Image Difference

Subtract the warped c1 from c2 to reveal circuit board differences.

In [ ]:
diff = cv.absdiff(c2, c1_warped)
diff_rgb = cv.cvtColor(diff, cv.COLOR_BGR2RGB)

# Enhance visibility
diff_enhanced = cv.normalize(diff, None, 0, 255, cv.NORM_MINMAX)
diff_enhanced_rgb = cv.cvtColor(diff_enhanced, cv.COLOR_BGR2RGB)

fig, ax = plt.subplots(1, 2, figsize=(14, 6))
ax[0].imshow(diff_rgb);          ax[0].set_title('Difference (raw)');      ax[0].axis('off')
ax[1].imshow(diff_enhanced_rgb); ax[1].set_title('Difference (enhanced)'); ax[1].axis('off')
plt.tight_layout()
plt.savefig('../output/p3b_diff.png', dpi=150)
plt.show()

## Part (c) — SIFT Feature Matching

Detect SIFT keypoints and descriptors in both images, then match with FLANN + Lowe's ratio test.

In [ ]:
g1 = cv.cvtColor(c1, cv.COLOR_BGR2GRAY)
g2 = cv.cvtColor(c2, cv.COLOR_BGR2GRAY)

sift = cv.SIFT_create()
kp1, des1 = sift.detectAndCompute(g1, None)
kp2, des2 = sift.detectAndCompute(g2, None)
print(f'Keypoints — c1: {len(kp1)}, c2: {len(kp2)}')

In [ ]:
# FLANN matcher
index_params  = dict(algorithm=1, trees=5)   # FLANN_INDEX_KDTREE = 1
search_params = dict(checks=50)
flann   = cv.FlannBasedMatcher(index_params, search_params)
matches = flann.knnMatch(des1, des2, k=2)

# Lowe's ratio test
good = []
matchesMask = [[0, 0] for _ in range(len(matches))]
for i, (m, n) in enumerate(matches):
    if m.distance < 0.7 * n.distance:
        good.append(m)
        matchesMask[i] = [1, 0]

print(f'Good matches: {len(good)} / {len(matches)}')

In [ ]:
draw_params = dict(matchColor=(0, 255, 0),
                   singlePointColor=(255, 0, 0),
                   matchesMask=matchesMask,
                   flags=cv.DrawMatchesFlags_DEFAULT)
img_matches = cv.drawMatchesKnn(g1, kp1, g2, kp2, matches, None, **draw_params)

plt.figure(figsize=(18, 7))
plt.imshow(img_matches)
plt.axis('off')
plt.title(f'SIFT Matches (good: {len(good)})')
plt.tight_layout()
plt.savefig('../output/p3c_sift_matches.png', dpi=150)
plt.show()

## Part (d) — Homography from Feature Matches

Compute homography from the SIFT matches using RANSAC inside `cv.findHomography`, then compare with the manual homography.

In [ ]:
src_pts = np.float32([kp1[m.queryIdx].pt for m in good]).reshape(-1, 1, 2)
dst_pts = np.float32([kp2[m.trainIdx].pt for m in good]).reshape(-1, 1, 2)

H_auto, mask_auto = cv.findHomography(src_pts, dst_pts, cv.RANSAC, ransacReprojThreshold=5.0)
inliers_auto = int(mask_auto.sum())
print(f'Auto Homography H (RANSAC inliers: {inliers_auto}/{len(good)}):')
print(H_auto)

In [ ]:
# Warp c1 using the auto homography
c1_warped_auto = cv.warpPerspective(c1, H_auto, (w, h))
c1_warped_auto_rgb = cv.cvtColor(c1_warped_auto, cv.COLOR_BGR2RGB)

fig, ax = plt.subplots(1, 3, figsize=(18, 6))
ax[0].imshow(c1_warped_rgb);      ax[0].set_title('Warp — Manual H');    ax[0].axis('off')
ax[1].imshow(c1_warped_auto_rgb); ax[1].set_title('Warp — Auto H (SIFT)'); ax[1].axis('off')
ax[2].imshow(c2_rgb);             ax[2].set_title('c2 (target)');         ax[2].axis('off')
plt.tight_layout()
plt.savefig('../output/p3d_comparison.png', dpi=150)
plt.show()

In [ ]:
# Quantitative comparison
print('Manual H:')
print(np.round(H_manual, 4))
print('\nAuto H:')
print(np.round(H_auto, 4))

if H_manual is not None and H_auto is not None:
    # Normalise both by H[2,2] before comparing
    H_m_n = H_manual / H_manual[2, 2]
    H_a_n = H_auto   / H_auto[2, 2]
    print('\nElement-wise difference (normalised):')
    print(np.round(H_m_n - H_a_n, 4))